# 02 Preprocessing Data

Notebook ini digunakan untuk membersihkan teks ulasan, melakukan
normalisasi bahasa Indonesia, dan menyiapkan dataset untuk analisis
sentimen serta topic modeling.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "bjb_reviews_raw.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "bjb_reviews_clean.csv"


## Langkah Preprocessing

- Hapus duplikasi dan nilai kosong
- Bersihkan URL, simbol, angka, dan noise teks
- Normalisasi beberapa kata tidak baku sederhana
- Hapus stopword bahasa Indonesia
- Lakukan stemming dengan Sastrawi jika tersedia
- Bentuk label otomatis dari skor review jika dibutuhkan


In [ ]:
import pandas as pd

from src.preprocessing import clean_text, infer_label_from_score, preprocess_text

df_raw = pd.read_csv(RAW_PATH)
df_raw.head()


In [ ]:
df = df_raw.copy()
df = df.drop_duplicates(subset=[col for col in ["review_id", "content"] if col in df.columns])
df["content"] = df["content"].fillna("").astype(str)
df = df[df["content"].str.strip() != ""].copy()

df["content_clean"] = df["content"].apply(preprocess_text)

if "label" not in df.columns:
    df["label"] = ""

mask_empty_label = df["label"].fillna("").astype(str).str.strip() == ""
df.loc[mask_empty_label, "label"] = df.loc[mask_empty_label, "score"].apply(infer_label_from_score)

df = df[df["content_clean"].str.strip() != ""].copy()
df.head()


In [ ]:
output_columns = ["review_id", "user_name", "score", "content", "content_clean", "label"]
available_columns = [column for column in output_columns if column in df.columns]
df[available_columns].to_csv(PROCESSED_PATH, index=False)
print(f"Saved preprocessing result to: {PROCESSED_PATH}")
df[available_columns].head()
